In [1]:
%matplotlib qt
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from operator import itemgetter
from PyPDF2 import PdfMerger

# plt.style.use('dark_background')

In [2]:
def mergePDFFiles(src_dir, tar_dir, tar_fname):
    '''
    Merges all PDF files in `src_dict` into one PDF file and stores it
    in `tar_dir` as `tar_fname`.
    '''
    # Find all PDF files
    pdfs = [os.path.join(src_dir, f) for f in os.listdir(src_dir) if f[-4:] == '.pdf']
    # Merge all found PDF files
    merger = PdfMerger()
    for pdf in pdfs:
        merger.append(pdf)
    # Save merged PDF file and close PdfMerger() object
    fpath_merged = os.path.join(tar_dir, tar_fname)
    merger.write(fpath_merged)
    merger.close()
    return None

In [3]:
# Given
compute_fmlt_mode = 'rest1'; compute_lt_rel_mode = 'rest1'
dir_results = os.path.expanduser('~/research/results/wash-u/')
fname_st_rel = 'shortterm_reliability_values.csv'
fname_lt_rel = 'longterm_reliability_values_compmode=' + compute_lt_rel_mode + '.csv'
fname_fmlt = 'familiality_values_compmode=' + compute_fmlt_mode + '.csv'
annot_size = 10

# Set plotting parameters
plt.rcParams['figure.figsize'] = [16, 7]
plt.rcParams['font.size'] = 14

# Create a colormap to have a fixed color to represent each feature
colors = cm.cividis_r(np.linspace(0, 1, 2))
marker_shapes = ['o', 's']

# Compare FFT-FOOOF measures with FFT and HHT

In [5]:
ftr_labels = {
    'F25': r'AP$_{\delta+\theta}$',
    'F26': r'AP$_\alpha$',
    'F27': r'AP$_\beta$',
    'F40': r'Dom. Freq. ($DF$)',
    'F41': r'AP$_{DF}$',
}
inds_ftrs = [24, 25, 26, 39, 40]
spec_modes = ['fft', 'fft-fooof', 'hht']; n_specmodes = len(spec_modes)

data_strels_specmode = {}  # data to be compared
for st_rel_cond in ['Age-18', 'Age-20', 'Mean']:
    for i_spec_mode, spec_mode in enumerate(spec_modes):

        # Get file path to the short-term reliability dataframe
        dir_rel = os.path.expanduser('~/research/results/wash-u/reliability/' + spec_mode)
        fpath_st_rel = os.path.join(dir_rel, fname_st_rel)

        # Read short-term reliability values
        df_st_rel = pd.read_csv(fpath_st_rel)

        # Populate extracted st-rel values
        data_strels_specmode[spec_mode] = [df_st_rel[st_rel_cond][i] for i in inds_ftrs]

    # Make and display dataframe of collected st-rel results
    df_strels_specmode = pd.DataFrame.from_dict(data_strels_specmode)
    df_strels_specmode.insert(loc=0, column='Feature', value=['F%d' % (i + 1) for i in inds_ftrs])
    print("Short-Term Reliability [%s]" % st_rel_cond)
    display(df_strels_specmode)

    # Visualize
    fpath_fig = os.path.join(dir_results, 'reliability',
                             'shortterm_reliability_comparison_FFTVsFFTFOOOFVsHHT_%s.pdf' % st_rel_cond)
    plt.figure()
    for i_ftr, ftr in enumerate(df_strels_specmode['Feature']):
        strels_specmode_ftr = df_strels_specmode.iloc[i_ftr].to_list()[1:]
        plt.plot(range(n_specmodes), strels_specmode_ftr, 'o-', label="%s: %s"%(ftr, ftr_labels[ftr]))
    plt.legend()
    plt.xlim([-1, 3])
    plt.ylim([0, 1])
    plt.ylabel('Reliability')
    plt.xticks(range(n_specmodes), [mode.upper() for mode in spec_modes])
    plt.title('Short-term reliability (%s)' % st_rel_cond)
    plt.savefig(fpath_fig, bbox_inches='tight')
    plt.savefig(fpath_fig[:-4] + '.png', bbox_inches='tight') # also save as png
    plt.show()

Short-Term Reliability [Age-18]


,Feature,fft,fft-fooof,hht
0,F25,0.109958,0.268965,0.723972
1,F26,0.936595,0.578115,0.939189
2,F27,0.607615,0.614420,0.686800
3,F40,0.791672,0.573493,0.633039
4,F41,0.543256,0.523697,0.048618


Short-Term Reliability [Age-20]


,Feature,fft,fft-fooof,hht
0,F25,0.340920,0.405247,0.746339
1,F26,0.724377,0.585341,0.727165
2,F27,0.706247,0.747912,0.714913
3,F40,0.531316,0.549325,0.562528
4,F41,0.546552,0.611459,0.536791


Short-Term Reliability [Mean]


,Feature,fft,fft-fooof,hht
0,F25,0.225439,0.337106,0.735155
1,F26,0.830486,0.581728,0.833177
2,F27,0.656931,0.681166,0.700856
3,F40,0.661494,0.561409,0.597784
4,F41,0.544904,0.567578,0.292705


# Sort short-term reliability values in descending order

In [ ]:
for st_rel_cond in ['Age-18', 'Age-21', 'Mean']:
# for st_rel_cond in ['Age-18']:
    plt.figure()
    for i_spec_mode, spec_mode in enumerate(['fft', 'hht']):
        # Get file path to the short-term reliability dataframe
        dir_rel = os.path.expanduser('~/research/results/wash-u/reliability/' + spec_mode)
        fpath_st_rel = os.path.join(dir_rel, fname_st_rel)

        # Read short-term reliability values
        df_st_rel = pd.read_csv(fpath_st_rel)

        # Create a dictionary of st-rel values for st_rel_cond (e.g. Age-18)
        st_rel_dict = {}
        for i, v in enumerate(df_st_rel[st_rel_cond]):
            if not pd.isna(v):
                st_rel_dict['F%d' % (i + 1)] = v

        # Sort st-rel values in the descending order
        st_rel_dict_sorted = dict(sorted(st_rel_dict.items(), key=itemgetter(1), reverse=True))

        # Plot and save sorted st-rel values using both spec modes
        plt.scatter(range(len(st_rel_dict)), st_rel_dict_sorted.values(), alpha=0.9, label=spec_mode,
                   marker=marker_shapes[i_spec_mode], color=colors[i_spec_mode])
        for i_Fs, Fs in enumerate(st_rel_dict_sorted):
            plt.annotate(Fs, (i_Fs, st_rel_dict_sorted[Fs]), size=annot_size)
    plt.legend()
    plt.axhline(y=0.5, color='r', ls='-')
    plt.ylim([-0.2, 1])
    plt.ylabel('Reliability')
    plt.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    plt.title('Short-term reliability (%s)' % st_rel_cond)
    fpath_fig = os.path.join(dir_results, 'reliability',
                             'shortterm_reliability_ordered_FFTVsHHT_%s.pdf' % st_rel_cond)
    plt.savefig(fpath_fig, bbox_inches='tight')
    plt.savefig(fpath_fig[:-4] + '.png', bbox_inches='tight') # also save as png
    plt.show()

# Sort long-term reliability values in descending order

In [ ]:
lt_rel_cond = 'Age-18Vs21'
plt.figure()
for i_spec_mode, spec_mode in enumerate(['fft', 'hht']):
    # Get file path to the long-term reliability dataframe
    dir_rel = os.path.expanduser('~/research/results/wash-u/reliability/' + spec_mode)
    fpath_lt_rel = os.path.join(dir_rel, fname_lt_rel)

    # Read long-term reliability values
    df_lt_rel = pd.read_csv(fpath_lt_rel)

    # Create a dictionary of lt-rel values for lt_rel_cond (i.e. Age-18Vs21)
    lt_rel_dict = {}
    for i, v in enumerate(df_lt_rel[lt_rel_cond]):
        if not pd.isna(v):
            lt_rel_dict['F%d' % (i + 1)] = v

    # Sort lt-rel values in the descending order
    lt_rel_dict_sorted = dict(sorted(lt_rel_dict.items(), key=itemgetter(1), reverse=True))

    # Plot and save sorted lt-rel values of all features
    plt.scatter(range(len(lt_rel_dict)), lt_rel_dict_sorted.values(), alpha=0.9, label=spec_mode,
                   marker=marker_shapes[i_spec_mode], color=colors[i_spec_mode])
    for i_Fs, Fs in enumerate(lt_rel_dict_sorted):
        plt.annotate(Fs, (i_Fs, lt_rel_dict_sorted[Fs]), size=annot_size)
plt.legend()
plt.axhline(y=0.5, color='r', ls='-')
plt.ylim([-0.2, 1])
plt.ylabel('Reliability')
plt.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
plt.title('Long-term reliability (%s)' % lt_rel_cond)
fpath_fig = os.path.join(dir_results, 'reliability',
                         'longterm_reliability_ordered_compmode=%s_FFTVsHHT_%s.pdf'
                         % (compute_lt_rel_mode, lt_rel_cond))
plt.savefig(fpath_fig, bbox_inches='tight')
plt.savefig(fpath_fig[:-4] + '.png', bbox_inches='tight') # also save as png
plt.show()

# Sort familiality values in descending order

In [ ]:
for fmlt_cond in ['Age-18', 'Age-21', 'Mean']:
# for fmlt_cond in ['Age-18']:
    plt.figure()
    for i_spec_mode, spec_mode in enumerate(['fft', 'hht']):
        # Get file path to the familiality dataframe
        dir_fmlt = os.path.expanduser('~/research/results/wash-u/familiality/' + spec_mode)
        fpath_fmlt = os.path.join(dir_fmlt, fname_fmlt)

        # Read familiality values
        df_fmlt = pd.read_csv(fpath_fmlt)

        # Create a dictionary of fmlt values for fmlt_cond (e.g. AgeGroup-4)
        fmlt_dict = {}
        for i, v in enumerate(df_fmlt[fmlt_cond]):
            if not pd.isna(v):
                fmlt_dict['F%d' % (i + 1)] = v

        # Sort fmlt values in the descending order
        fmlt_dict_sorted = dict(sorted(fmlt_dict.items(), key=itemgetter(1), reverse=True))

        # Plot and save sorted familiality values of all features
        plt.scatter(range(len(fmlt_dict)), fmlt_dict_sorted.values(), alpha=0.9, label=spec_mode,
                   marker=marker_shapes[i_spec_mode], color=colors[i_spec_mode])
        for i_Fs, Fs in enumerate(fmlt_dict_sorted):
            plt.annotate(Fs, (i_Fs, fmlt_dict_sorted[Fs]), size=annot_size)
    plt.legend()
    plt.axhline(y=0.5, color='r', ls='-')
    plt.ylim([-0.2, 1])
    plt.ylabel('Familiality')
    plt.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    plt.title('Familiality (%s)' % fmlt_cond)
    fpath_fig = os.path.join(dir_results, 'familiality', 'familiality_ordered_compmode=%s_FFTVsHHT_%s.pdf' %
                             (compute_fmlt_mode, fmlt_cond))
    plt.savefig(fpath_fig, bbox_inches='tight')
    plt.savefig(fpath_fig[:-4] + '.png', bbox_inches='tight') # also save as png
    plt.show()